## Matching distance 

This is the notebook where we will merge all the code to compute the matching distance

In [1]:
import gudhi
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
import numpy as np
import matplotlib
import matplotlib.patches as mpatches

from matplotlib import rc

# rc('font',**{'family':'sans-serif','sans-serif':['Helvetica']})
rc("font", **{"family": "serif", "serif": ["Times"]})
rc("text", usetex=True)


from math import cos, sin, pi, atan     
import itertools
from scipy.stats import qmc
import math

import numpy.linalg
import random as rnd

## for checking if in convex hull
import scipy.spatial
from scipy.optimize import linprog

import sys 
sys.path.append('../Code/')
import modules as mod
import examples as ex
import plot as ourplot
import csv
import bunny_smaller
import omega_pts as omega
# This reloads external modules every time a cell is run
# so we don't have to reboot kernel every time we modify classes.py

%load_ext autoreload
%autoreload 2

## 0. Let's load an example to make our lives easier for now

In [2]:
critical_vals, nolub = ex.small_socg_ex()
print(critical_vals)

[(parent module of (0.0, 0.0) is 0), (parent module of (6.0, 0.0) is 0), (parent module of (0.0, 6.0) is 0), (parent module of (4.0, 2.0) is 1), (parent module of (4.0, 4.0) is 1), (parent module of (8.0, 2.0) is 1), (parent module of (1.0, 5.0) is 1), (parent module of (1.0, 10.0) is 1), (parent module of (2.0, 5.0) is 1)]


## 1. Determine the sets of critical values $C_M$ and $C_N$ for $M$ and $N$, respectively 

##### so far we have only considered rectangle modules which are given directly by their vertices so we know the critical values fom the start (??)

In [3]:
# put the modules into the correct format (eg like in examples.py)
critval_coords = []
for i in range(len(critical_vals)):
    critval_coords.append(critical_vals[i].coords)
print(critval_coords)

[(0.0, 0.0), (6.0, 0.0), (0.0, 6.0), (4.0, 2.0), (4.0, 4.0), (8.0, 2.0), (1.0, 5.0), (1.0, 10.0), (2.0, 5.0)]


## 2. Identify the closures $\overline{C_M}$ and $\overline{C_N}$ 

In [4]:
## we have code to compute the closure under least upper bound in RectangleModules class 
## and also some code to compute the closure under intersection in omega_pts.py!!!! 
## adapt the code to have critical values in the good format for the switch pts functions

# NOTE: this function takes in the actual crit vals (in the having-a-parent-format)!
# so we need to input that, not just the coords we just stripped out.
print("hellooo I am an answer and maybe I am right?")
lub = omega.closure_lub(critical_vals)
for i in range(len(lub)):
    print(lub[i])


hellooo I am an answer and maybe I am right?
[(6.0, 0.0), (0.0, 6.0), (6.0, 0.0), (6.0, 6.0), (0.0, 6.0), (6.0, 6.0)]
[(4.0, 4.0), (8.0, 2.0), (4.0, 5.0), (4.0, 10.0), (4.0, 5.0), (4.0, 4.0), (8.0, 4.0), (4.0, 5.0), (4.0, 10.0), (4.0, 5.0), (8.0, 2.0), (8.0, 4.0), (8.0, 5.0), (8.0, 10.0), (8.0, 5.0), (4.0, 5.0), (4.0, 5.0), (8.0, 5.0), (1.0, 10.0), (2.0, 5.0), (4.0, 10.0), (4.0, 10.0), (8.0, 10.0), (1.0, 10.0), (2.0, 10.0), (4.0, 5.0), (4.0, 5.0), (8.0, 5.0), (2.0, 5.0), (2.0, 10.0)]


## 3. Compute the set of switch-points $\Omega=\Omega(M,N)$ using the algorithms 

In [14]:
### run all three allgoriths 3v1, 2v2 paired , 2v2 unpaired
### copied and pasted from socg_examples.ipynb


## For now: rename. Later: rewrite. 
critpts, crits_no_lub = critical_vals, nolub
makegraphyesno = False
title = 'title'

############################
if makegraphyesno:
    title = "M and N"
    dpi = 300

    ############ make the example
    fig, ax0 = plt.subplots()
    fig.set_dpi(dpi)
    fig.set_size_inches(6, 5)
    # fig.tight_layout

    fig.suptitle("Input rectangle modules M and N")
    for i in [0, 3, 6]:
        p1, p2, p3 = crits_no_lub[0 + i][0], crits_no_lub[1 + i][0], crits_no_lub[2 + i][0]
        p4 = np.array(p2) + np.array(p3) - np.array(p1)
        x = [p1[0], p2[0], p4[0], p3[0], p1[0]]
        y = [p1[1], p2[1], p4[1], p3[1], p1[1]]
        ax0.plot(x, y, "k--")

    for c in crits_no_lub:
        if c[1] == 0:
            ax0.plot(
                c[0][0],
                c[0][1],
                "o",
                color="black",
                label="black",
                markersize=4,
            )
        else:
            ax0.plot(
                c[0][0],
                c[0][1],
                "o",
                color="black",
                markersize=4,
            )
            ax0.plot(
                c[0][0],
                c[0][1],
                "o",
                color="orange",
                label="orange",
                markersize=3,
            )


    black_patch = mpatches.Patch(color="black", label="critical values from M")
    orange_patch = mpatches.Patch(color="orange", label="critical values from N")
    ax0.legend(
        handles=[black_patch, orange_patch],
        loc="upper right",
    )
    plt.show()


############# do computations
# algo 3v1 = Algorithm 3 in SoCG overleaf
# never has points at infinity, so we can plot all points
print("3v1")
omegapts_3v1, _, _ = omega.output_alg_3_1(critpts, "socg/" + title)
if makegraphyesno:
    ourplot.plot_both_c_o(crits_no_lub, omegapts_3v1, title, dpi, "3v1")

# algo 2v2 paired = Algorithm 5 in SoCG overleaf
print("2v2paired")
omegapts_2v2_p, _, _, is_inf_2v2p, orderedomegas2v2p = omega.output_alg_2_2_paired(
    critpts, "socg/" + title
)

if makegraphyesno:
    ourplot.plot_c_o_with_inf(crits_no_lub, orderedomegas, is_inf, title, dpi, "2v2 Paired")

# algo 2v2 unpaired = Algorithm 7 in SoCG overleaf
print("2v2unp")
omegapts_2v2_unp, _, _, is_inf_2v2unp, orderedomegas_2v2unp = omega.output_alg_2_2_unpaired(
    critpts, "socg/" + title
)
if makegraphyesno:
    ourplot.plot_c_o_with_inf(
        crits_no_lub, orderedomegas2, is_inf2, title, dpi, "2v2 Unpaired"
    )

3v1
**********
Computed 914 points (with duplicates)
Without duplicates 142
2v2paired
**********
Computed 133 points (with duplicates)
Without duplicates 44
2v2unp
**********
Computed 181 points (with duplicates)
Without duplicates 95


In [15]:
print(is_inf_2v2unp)

['m', None, None, None, None, None, None, None, 2.0, 2.5, 5.0, None, None, None, None, None, None, None, 0.8333333333333334, 0.5, 0.25, None, None, 0.25, 0.125, None, None, 0.5, None, None, None, None, None, None, None, None, None, None, 2.0, None, None, None, 1.0, None, None, 0.8, None, 1.0, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, None, 1.25, None, None, 0.8333333333333334, None]


## 3.5: Now we should put all the 3 algos outputs together into one big omega, and deal with slopes vs points.

In [26]:
omega_points_collection = []
omega_slopes_collection = []
omega_points_collection.append(omegapts_3v1)
# probs just the slopes for 2v2 p
omega_slopes_collection.append(is_inf_2v2p)

# decide whether slopes or points gets the thing
for thing in range(len(orderedomegas_2v2unp)):
    if is_inf_2v2unp == None: 
        omega_points_collection.append(is_inf_2v2unp[thing])
    else:
        omega_slopes_collection.append(orderedomegas_2v2unp[thing])

print(omega_points_collection)
print(omega_slopes_collection)

[[(12.0, 4.0), (5.5, 6.0), (5.0, 10.0), (8.0, 9.0), (10.0, 6.0), (8.0, 18.0), (2.0, 11.0), (4.0, 30.0), (6.0, 2.0), (6.0, 2.5), (2.5, 10.0), (7.5, 6.0), (6.0, 11.0), (6.0, 11.5), (7.0, 10.0), (18.0, 10.0), (6.0, 20.0), (4.0, 2.5), (6.5, 10.0), (3.0, 6.0), (8.0, 2.5), (4.5, 10.0), (8.0, 11.0), (9.0, 10.0), (0.5, 6.0), (6.0, 4.0), (16.0, 6.0), (6.0, 4.5), (6.0, 13.0), (2.0, 25.0), (8.0, 4.5), (0.0, 0.5), (4.0, 25.0), (1.5, 6.0), (6.0, 6.5), (8.0, 25.0), (9.5, 10.0), (3.5, 6.0), (3.0, 10.0), (14.0, 10.0), (0.0, 2.0), (0.0, 2.5), (4.0, 18.0), (16.0, 10.0), (2.0, 20.0), (20.0, 10.0), (12.0, 6.0), (4.0, 11.0), (4.0, 20.0), (6.0, 1.0), (8.0, 20.0), (6.0, 1.5), (2.0, 13.0), (14.0, 5.0), (4.0, 13.0), (5.5, 10.0), (8.0, 13.0), (10.0, 10.0), (2.0, 6.0), (16.0, 5.0), (8.0, 22.0), (20.0, 5.0), (7.5, 10.0), (6.0, 15.0), (12.0, 10.0), (4.0, 6.0), (4.0, 15.0), (8.0, 6.0), (8.0, 6.5), (1.0, 0.0), (8.0, 15.0), (2.0, 8.5), (8.0, 24.0), (2.0, 8.0), (6.0, 8.0), (6.0, 8.5), (6.0, 26.0), (5.0, 6.0), (4.0, 8.

## 4. For each  line $L$ with positive slope that:

- passes through pairs of points in $\overline{C_M}\cup \overline{C_N}$, or
- passes through one point in $\overline{C_M}\cup \overline{C_N}$ and one point in ${\Omega}$,   or 
- passes through  one point of $\overline{C_M}\cup \overline{C_N}\cup{\Omega}$ and has diagonal slope

do 

a. Compute the persistence diagrams $dgm(M^L)$, $dgm(N^L)$ of the restrictions of $M$ and $N$ to that line $L$;


b. Compute their weighted bottleneck distance $\hat m^L d_B(dgm(M^L),dgm(N^L))$


#### We need clarification in 4. 

In [38]:
CMuCN = lub
OMEGA = omega_collection

## 5. Output the maximum of the weighted bottleneck distances computed in (b). 